## Overall Introduction of Innovative Methods

This part extends the official baseline (fixed train-test split) by testing four methods for imbalanced text classification:

1. **SMOTE + Logistic Regression** — data balancing  
2. **Weighted Ensemble(Random Forest + Gradient Boosting Tree + Logistic Regression)** — model combination  
3. **Threshold Moving** — decision threshold adjustment  
4. **Confidence-Based Pseudo-Labeling** — semi-supervised learning  

The purpose is not only to seek performance improvement, but also to examine whether these strategies can better handle class imbalance and improve minority-class detection.

In [ ]:
! pip install pyspark

In [ ]:
!pip install dataproc-spark-connect==0.9.0 pyspark spark-nlp matplotlib graphframes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.8/317.8 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.8/766.8 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 12.6 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.8-py2.py3-none-any.whl size=318352946 sha256=0ba132184e829335ea98839eb446a278241d0037ad3f6e689acacd3aca742ab7
  Stored in directory: /root/.cache/pip/wheels/f0/f6/86/a9231691706c40d5bcc8c907f583e0ef90c075dcfa97e272d0
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
  

In [ ]:
# =========================
#  Imports
# =========================
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.linalg import Vectors, VectorUDT

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE

In [ ]:
# =========================
#  Create Spark Session
# =========================
spark = SparkSession.builder \
    .appName("CDS527_Innovation_Methods_YangYousen") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

SEED = 42

In [ ]:
# =========================
# Experiment Setup and Baseline
# =========================

# =========================
# 1. Load fixed split files
# =========================
train_path = "train_fixed_split.csv"
test_path = "test_fixed_split.csv"

train_df = spark.read.csv(train_path, header=True, inferSchema=True)
test_df = spark.read.csv(test_path, header=True, inferSchema=True)

print("=" * 80)
print("Loaded fixed split files")
print("=" * 80)
print("Train size:", train_df.count())
print("Test size:", test_df.count())

# Ensure schema
train_df = train_df.select(
    col("text").cast("string"),
    col("label").cast("double")
)

test_df = test_df.select(
    col("text").cast("string"),
    col("label").cast("double")
)

# =========================
# 2. Build PySpark TF-IDF features
#    Note:
#    - The fixed split is the same as baseline
#    - Features are implemented mainly in PySpark
# =========================
tokenizer = Tokenizer(inputCol="text", outputCol="words")
vectorizer = CountVectorizer(
    inputCol="words",
    outputCol="raw_features",
    vocabSize=10000,
    minDF=2.0
)
idf = IDF(inputCol="raw_features", outputCol="features")

feature_pipeline = Pipeline(stages=[tokenizer, vectorizer, idf])
feature_model = feature_pipeline.fit(train_df)

train_feat = feature_model.transform(train_df).select("text", "label", "features")
test_feat = feature_model.transform(test_df).select("text", "label", "features")

print("\nFeature engineering completed.")
print("Train feature count:", train_feat.count())
print("Test feature count:", test_feat.count())

# =========================
# 3. Helper functions
# =========================
def print_section_title(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def extract_predictions_from_spark(pred_df, label_col="label", pred_col="prediction"):
    pdf = pred_df.select(label_col, pred_col).toPandas()
    y_true = pdf[label_col].astype(int).values
    y_pred = pdf[pred_col].astype(int).values
    return y_true, y_pred

def evaluate_and_print(method_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    print_section_title(method_name)
    print("Test Accuracy:", round(acc, 4))
    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["negative", "positive"],
        digits=4
    ))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return acc

def get_metrics_dict(method_name, y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=["negative", "positive"],
        output_dict=True,
        digits=4
    )
    cm = confusion_matrix(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)

    return {
        "Method": method_name,
        "Accuracy": round(acc, 4),
        "Neg_Precision": round(report["negative"]["precision"], 4),
        "Neg_Recall": round(report["negative"]["recall"], 4),
        "Neg_F1": round(report["negative"]["f1-score"], 4),
        "Pos_Precision": round(report["positive"]["precision"], 4),
        "Pos_Recall": round(report["positive"]["recall"], 4),
        "Pos_F1": round(report["positive"]["f1-score"], 4),
        "Macro_F1": round(report["macro avg"]["f1-score"], 4),
        "Weighted_F1": round(report["weighted avg"]["f1-score"], 4),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1])
    }

# =========================
# 4. Add official baseline results from data_preprocess.py
# =========================
results = []

official_baseline = {
    "Method": "Official_Baseline_data_preprocess.py",
    "Accuracy": 0.9262,
    "Neg_Precision": 0.57,
    "Neg_Recall": 0.33,
    "Neg_F1": 0.42,
    "Pos_Precision": 0.94,
    "Pos_Recall": 0.98,
    "Pos_F1": 0.96,
    "Macro_F1": 0.69,
    "Weighted_F1": 0.92,
    "TN": 8,
    "FP": 16,
    "FN": 6,
    "TP": 268
}

results.append(official_baseline)

print_section_title("Official Baseline from data_preprocess.py")
print("Test Accuracy: 0.9262")
print("\nClassification Report:")
print("""
              precision    recall  f1-score   support

    negative       0.57      0.33      0.42        24
    positive       0.94      0.98      0.96       274

    accuracy                           0.93       298
   macro avg       0.76      0.66      0.69       298
weighted avg       0.91      0.93      0.92       298
""")
print("Confusion Matrix:")
print(np.array([[8, 16],
                [6, 268]]))

Loaded fixed split files
Train size: 1190
Test size: 298

Feature engineering completed.
Train feature count: 1190
Test feature count: 298

Official Baseline from data_preprocess.py
Test Accuracy: 0.9262

Classification Report:

              precision    recall  f1-score   support

    negative       0.57      0.33      0.42        24
    positive       0.94      0.98      0.96       274

    accuracy                           0.93       298
   macro avg       0.76      0.66      0.69       298
weighted avg       0.91      0.93      0.92       298

Confusion Matrix:
[[  8  16]
 [  6 268]]


In [ ]:
# =========================
#    Innovation 1: SMOTE + Logistic Regression
#    Main pipeline stays in PySpark
#    SMOTE is applied on training feature vectors
# =========================
print_section_title("Innovation 1 - SMOTE + Logistic Regression")

# Convert training Spark features to numpy for SMOTE
train_pd = train_feat.select("label", "features").toPandas()
X_train_np = np.array([x.toArray() for x in train_pd["features"]])
y_train_np = train_pd["label"].astype(int).values

print("Class distribution before SMOTE:")
unique, counts = np.unique(y_train_np, return_counts=True)
print(dict(zip(unique, counts)))

# Apply SMOTE
smote = SMOTE(random_state=SEED)
X_resampled, y_resampled = smote.fit_resample(X_train_np, y_train_np)

print("Class distribution after SMOTE:")
unique_res, counts_res = np.unique(y_resampled, return_counts=True)
print(dict(zip(unique_res, counts_res)))

# Back to Spark DataFrame (convert ndarray -> list)
smote_pd = pd.DataFrame({
    "label": y_resampled.tolist(),
    "features_array": [x.tolist() for x in X_resampled]  # 修复点
})

smote_spark = spark.createDataFrame(smote_pd)

# Convert list back to Spark MLlib Vector
array_to_vector_udf = udf(lambda arr: Vectors.dense(arr), VectorUDT())

smote_train_feat = smote_spark.withColumn(
    "features", array_to_vector_udf(col("features_array"))
).select(
    col("label").cast("double"),
    col("features")
)

# Logistic Regression model
lr_smote = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100
)

# Train and predict
smote_model = lr_smote.fit(smote_train_feat)
smote_pred = smote_model.transform(test_feat)

# Extract predictions and evaluate
y_true_smote, y_pred_smote = extract_predictions_from_spark(smote_pred)
evaluate_and_print("Innovation 1 - SMOTE + Logistic Regression", y_true_smote, y_pred_smote)

# Save results
results.append(get_metrics_dict("Innovation1_SMOTE_LR", y_true_smote, y_pred_smote))



Innovation 1 - SMOTE + Logistic Regression
Class distribution before SMOTE:
{0: 94, 1: 1096}
Class distribution after SMOTE:
{0: 1096, 1: 1096}

Innovation 1 - SMOTE + Logistic Regression
Test Accuracy: 0.9161

Classification Report:
              precision    recall  f1-score   support

    negative     0.4615    0.2500    0.3243        24
    positive     0.9368    0.9745    0.9553       274

    accuracy                         0.9161       298
   macro avg     0.6992    0.6122    0.6398       298
weighted avg     0.8986    0.9161    0.9045       298

Confusion Matrix:
[[  6  18]
 [  7 267]]


In [ ]:
# =========================
#  Innovation 2: Weighted Ensemble
#    RF + GBT + LR
# =========================
print_section_title("Innovation 2 - Weighted Ensemble (RF + GBT + LR)")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="rf_prediction",
    probabilityCol="rf_probability",
    numTrees=120,
    maxDepth=8,
    seed=SEED
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="gbt_prediction",
    maxIter=60,
    maxDepth=5,
    seed=SEED
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="lr_prediction",
    probabilityCol="lr_probability",
    maxIter=100
)

# Train models
rf_model = rf.fit(train_feat)
gbt_model = gbt.fit(train_feat)
lr_model = lr.fit(train_feat)

# Apply models one by one, renaming rawPrediction after each step
ensemble_df = test_feat

ensemble_df = rf_model.transform(ensemble_df)
ensemble_df = ensemble_df.withColumnRenamed("rawPrediction", "rf_rawPrediction")

ensemble_df = gbt_model.transform(ensemble_df)
ensemble_df = ensemble_df.withColumnRenamed("rawPrediction", "gbt_rawPrediction")

ensemble_df = lr_model.transform(ensemble_df)
ensemble_df = ensemble_df.withColumnRenamed("rawPrediction", "lr_rawPrediction")

# Extract positive class probability
def get_pos_prob(v):
    return float(v[1])

get_pos_prob_udf = udf(get_pos_prob, DoubleType())

# For GBT, convert rawPrediction -> probability using sigmoid
import math
def raw_to_prob(raw_vec):
    margin = float(raw_vec[1])
    return 1.0 / (1.0 + math.exp(-margin))

raw_to_prob_udf = udf(raw_to_prob, DoubleType())

ensemble_df = ensemble_df \
    .withColumn("rf_pos_prob", get_pos_prob_udf(col("rf_probability"))) \
    .withColumn("lr_pos_prob", get_pos_prob_udf(col("lr_probability"))) \
    .withColumn("gbt_pos_prob", raw_to_prob_udf(col("gbt_rawPrediction")))

# Weighted probability
w_rf = 0.35
w_gbt = 0.40
w_lr = 0.25

ensemble_df = ensemble_df.withColumn(
    "ensemble_prob",
    w_rf * col("rf_pos_prob") + w_gbt * col("gbt_pos_prob") + w_lr * col("lr_pos_prob")
)

# Convert probability to prediction
prob_to_pred_udf = udf(lambda p: 1 if p >= 0.5 else 0, IntegerType())

ensemble_df = ensemble_df.withColumn(
    "prediction",
    prob_to_pred_udf(col("ensemble_prob"))
)

# Evaluate
y_true_ens, y_pred_ens = extract_predictions_from_spark(ensemble_df)
evaluate_and_print("Innovation 2 - Weighted Ensemble (RF + GBT + LR)", y_true_ens, y_pred_ens)

results.append(get_metrics_dict("Innovation2_Weighted_Ensemble", y_true_ens, y_pred_ens))



Innovation 2 - Weighted Ensemble (RF + GBT + LR)

Innovation 2 - Weighted Ensemble (RF + GBT + LR)
Test Accuracy: 0.9094

Classification Report:
              precision    recall  f1-score   support

    negative     0.2000    0.0417    0.0690        24
    positive     0.9215    0.9854    0.9524       274

    accuracy                         0.9094       298
   macro avg     0.5608    0.5135    0.5107       298
weighted avg     0.8634    0.9094    0.8812       298

Confusion Matrix:
[[  1  23]
 [  4 270]]


In [ ]:
# =========================
#    Innovation 3: Threshold Moving
#    Find best threshold on validation split
# =========================
print_section_title("Innovation 3 - Threshold Moving")

train_sub, valid_sub = train_feat.randomSplit([0.8, 0.2], seed=SEED)

lr_threshold = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100
)

threshold_model = lr_threshold.fit(train_sub)
valid_pred = threshold_model.transform(valid_sub)

valid_pdf = valid_pred.select("label", "probability").toPandas()
valid_pdf["pos_prob"] = valid_pdf["probability"].apply(lambda v: float(v[1]))

best_threshold = 0.50
best_macro_f1 = -1

for threshold in np.arange(0.30, 0.71, 0.02):
    y_valid_true = valid_pdf["label"].astype(int).values
    y_valid_pred = (valid_pdf["pos_prob"] >= threshold).astype(int)
    macro_f1 = f1_score(y_valid_true, y_valid_pred, average="macro")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_threshold = threshold

print(f"Best threshold found on validation set: {best_threshold:.2f}")
print(f"Best validation macro F1: {best_macro_f1:.4f}")

# Retrain using full training set
final_threshold_model = lr_threshold.fit(train_feat)
test_threshold_pred = final_threshold_model.transform(test_feat)

test_threshold_pdf = test_threshold_pred.select("label", "probability").toPandas()
test_threshold_pdf["pos_prob"] = test_threshold_pdf["probability"].apply(lambda v: float(v[1]))
test_threshold_pdf["prediction"] = (test_threshold_pdf["pos_prob"] >= best_threshold).astype(int)

y_true_threshold = test_threshold_pdf["label"].astype(int).values
y_pred_threshold = test_threshold_pdf["prediction"].astype(int).values

evaluate_and_print(
    f"Innovation 3 - Threshold Moving (threshold={best_threshold:.2f})",
    y_true_threshold,
    y_pred_threshold
)

results.append(get_metrics_dict("Innovation3_Threshold_Moving", y_true_threshold, y_pred_threshold))


Innovation 3 - Threshold Moving
Best threshold found on validation set: 0.30
Best validation macro F1: 0.5605

Innovation 3 - Threshold Moving (threshold=0.30)
Test Accuracy: 0.7483

Classification Report:
              precision    recall  f1-score   support

    negative     0.1772    0.5833    0.2718        24
    positive     0.9543    0.7628    0.8479       274

    accuracy                         0.7483       298
   macro avg     0.5658    0.6731    0.5599       298
weighted avg     0.8918    0.7483    0.8015       298

Confusion Matrix:
[[ 14  10]
 [ 65 209]]


In [ ]:
# =========================
# Innovation 4: Confidence-Based Pseudo-Labeling
# =========================
print_section_title("Innovation - Confidence-Based Pseudo-Labeling")

from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.sql.functions import col, udf, lit
from pyspark.sql.types import DoubleType
import numpy as np

# --------------------------------------------------
#  Simulate unlabeled pool from train set
# --------------------------------------------------
labeled_df, pseudo_pool_df = train_df.randomSplit([0.8, 0.2], seed=SEED)

print("labeled_df size:", labeled_df.count())
print("pseudo_pool_df size:", pseudo_pool_df.count())
print("test_df size:", test_df.count())

# --------------------------------------------------
#  TF-IDF feature pipeline
# --------------------------------------------------
regex_tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="tokens",
    pattern="\\W+",
    toLowercase=True
)

stopword_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    vocabSize=20000,
    minDF=2.0
)

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

pipeline = Pipeline(stages=[
    regex_tokenizer,
    stopword_remover,
    vectorizer,
    idf
])

tfidf_model = pipeline.fit(labeled_df)

labeled_feat = tfidf_model.transform(labeled_df).select("text", "label", "features")
pseudo_pool_feat = tfidf_model.transform(pseudo_pool_df).select("text", "features")
test_feat = tfidf_model.transform(test_df).select("label", "features")

# --------------------------------------------------
#  Train initial classifier
# --------------------------------------------------
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

initial_model = lr.fit(labeled_feat)

# --------------------------------------------------
#  Predict pseudo labels for unlabeled pool
# --------------------------------------------------
pseudo_pred = initial_model.transform(pseudo_pool_feat)

extract_prob_udf = udf(lambda v: float(v[1]), DoubleType())
pseudo_pred = pseudo_pred.withColumn("pos_prob", extract_prob_udf(col("probability")))

# --------------------------------------------------
#  Confidence filtering
# --------------------------------------------------
POS_THRESHOLD = 0.95
NEG_THRESHOLD = 0.05

high_conf_pos = pseudo_pred.filter(col("pos_prob") >= POS_THRESHOLD) \
    .select("text", "features") \
    .withColumn("final_label", lit(1.0))

high_conf_neg = pseudo_pred.filter(col("pos_prob") <= NEG_THRESHOLD) \
    .select("text", "features") \
    .withColumn("final_label", lit(0.0))

pseudo_selected = high_conf_pos.unionByName(high_conf_neg)

print("Selected pseudo-labeled samples:", pseudo_selected.count())
print("Pseudo positive count:", high_conf_pos.count())
print("Pseudo negative count:", high_conf_neg.count())

# --------------------------------------------------
#  Build augmented training set
# --------------------------------------------------
original_labeled_train = labeled_feat.select(
    "text",
    "features",
    col("label").alias("final_label")
)

augmented_train = original_labeled_train.unionByName(pseudo_selected)

print("Original labeled training size:", original_labeled_train.count())
print("Augmented training size:", augmented_train.count())

# --------------------------------------------------
#  Retrain on augmented data
# --------------------------------------------------
pseudo_lr = LogisticRegression(
    featuresCol="features",
    labelCol="final_label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

pseudo_model = pseudo_lr.fit(augmented_train)

# --------------------------------------------------
#  Evaluate on test set
# --------------------------------------------------
test_eval_df = test_feat.select(
    "features",
    col("label").alias("final_label")
)

test_pred = pseudo_model.transform(test_eval_df).select("final_label", "prediction")

rows = test_pred.collect()
y_true = [row["final_label"] for row in rows]
y_pred = [row["prediction"] for row in rows]

evaluate_and_print(
    f"Innovation - Confidence-Based Pseudo-Labeling ({POS_THRESHOLD:.2f}/{NEG_THRESHOLD:.2f})",
    y_true,
    y_pred
)

results.append(
    get_metrics_dict(
        f"Innovation4_PseudoLabeling",
        y_true,
        y_pred
    )
)


Innovation - Confidence-Based Pseudo-Labeling
labeled_df size: 982
pseudo_pool_df size: 208
test_df size: 298
Selected pseudo-labeled samples: 203
Pseudo positive count: 156
Pseudo negative count: 47
Original labeled training size: 982
Augmented training size: 1185

Innovation - Confidence-Based Pseudo-Labeling (0.95/0.05)
Test Accuracy: 0.7483

Classification Report:
              precision    recall  f1-score   support

    negative     0.1928    0.6667    0.2991        24
    positive     0.9628    0.7555    0.8466       274

    accuracy                         0.7483       298
   macro avg     0.5778    0.7111    0.5728       298
weighted avg     0.9008    0.7483    0.8025       298

Confusion Matrix:
[[ 16   8]
 [ 67 207]]


In [ ]:
# =========================
#    Final comparison table
# =========================
print_section_title("Final Comparison Table: Official Baseline vs Innovation Methods")

results_df = pd.DataFrame(results)

# Remove duplicates to avoid the same method appearing multiple times
results_df = results_df.drop_duplicates(subset=["Method"]).reset_index(drop=True)

# Sort by Macro_F1 first, then Accuracy
results_df = results_df.sort_values(by=["Macro_F1", "Accuracy"], ascending=False).reset_index(drop=True)

print(results_df)

# Save result table
results_df.to_csv("innovation_vs_baseline_results.csv", index=False, encoding="utf-8-sig")
print("\nSaved file: innovation_vs_baseline_results.csv")

# Optional display for Colab
try:
    from IPython.display import display
    display(results_df)
except:
    pass

# =========================
#  Short summary text for notebook
# =========================
print_section_title("Short Summary")
best_method = results_df.iloc[0]["Method"]
best_macro_f1 = results_df.iloc[0]["Macro_F1"]
best_acc = results_df.iloc[0]["Accuracy"]

print(f"The best-performing method in this comparison is: {best_method}")
print(f"Best Macro-F1: {best_macro_f1}")
print(f"Best Accuracy: {best_acc}")



Final Comparison Table: Official Baseline vs Innovation Methods
                                 Method  Accuracy  Neg_Precision  Neg_Recall  \
0  Official_Baseline_data_preprocess.py    0.9262         0.5700      0.3300   
1                  Innovation1_SMOTE_LR    0.9161         0.4615      0.2500   
2            Innovation4_PseudoLabeling    0.7483         0.1928      0.6667   
3          Innovation3_Threshold_Moving    0.7483         0.1772      0.5833   
4         Innovation2_Weighted_Ensemble    0.9094         0.2000      0.0417   

   Neg_F1  Pos_Precision  Pos_Recall  Pos_F1  Macro_F1  Weighted_F1  TN  FP  \
0  0.4200         0.9400      0.9800  0.9600    0.6900       0.9200   8  16   
1  0.3243         0.9368      0.9745  0.9553    0.6398       0.9045   6  18   
2  0.2991         0.9628      0.7555  0.8466    0.5728       0.8025  16   8   
3  0.2718         0.9543      0.7628  0.8479    0.5599       0.8015  14  10   
4  0.0690         0.9215      0.9854  0.9524    0.5107     

,Method,Accuracy,Neg_Precision,Neg_Recall,Neg_F1,Pos_Precision,Pos_Recall,Pos_F1,Macro_F1,Weighted_F1,TN,FP,FN,TP
0,Official_Baseline_data_preprocess.py,0.9262,0.5700,0.3300,0.4200,0.9400,0.9800,0.9600,0.6900,0.9200,8,16,6,268
1,Innovation1_SMOTE_LR,0.9161,0.4615,0.2500,0.3243,0.9368,0.9745,0.9553,0.6398,0.9045,6,18,7,267
2,Innovation4_PseudoLabeling,0.7483,0.1928,0.6667,0.2991,0.9628,0.7555,0.8466,0.5728,0.8025,16,8,67,207
3,Innovation3_Threshold_Moving,0.7483,0.1772,0.5833,0.2718,0.9543,0.7628,0.8479,0.5599,0.8015,14,10,65,209
4,Innovation2_Weighted_Ensemble,0.9094,0.2000,0.0417,0.0690,0.9215,0.9854,0.9524,0.5107,0.8812,1,23,4,270



Short Summary
The best-performing method in this comparison is: Official_Baseline_data_preprocess.py
Best Macro-F1: 0.69
Best Accuracy: 0.9262


## Final Results Discussion

The final results show that all four methods performed worse than the official baseline.  

- **Baseline**: Best overall, with Accuracy = 0.9262 and Macro-F1 = 0.6900.  
- **SMOTE + Logistic Regression**: Closest to baseline, but still lower.  
- **Weighted Ensemble, Threshold Moving, Pseudo-Labeling**: Larger drops in accuracy or class balance.  

This indicates that the baseline was already well suited to the dataset, while the new methods introduced noise, added complexity, or over-adjusted toward minority classes. Although none of the four surpassed the baseline, the exploration remains valuable:  
- It clarifies the limitations of oversampling, ensembling, threshold adjustment, and pseudo-labeling under this feature space and class distribution.  
- It shows that innovation is not only about outperforming baselines, but also about testing ideas, identifying trade-offs, and informing more targeted future improvements.
